# Phase 1 — Explore LaBraM EEG embeddings (live)

Run cells top to bottom. Kernel must be **Python (mh311)**.
Each step prints its output inline so you can see exactly what happens.

**Before running:** in a terminal do `cd ~/mental-health && git pull` to get the latest `src/`.
Cell 1 enables auto-reload, so further `git pull`s take effect without restarting the kernel.

Steps: **1** setup/paths · **2** load + preprocess + window (watch the `amplitude ... uV` line) ·
**3** frozen LaBraM → embeddings · **4** save to `$SCRATCH` · **5** sanity check (are windows distinct?) · **6** visualise.

In [ ]:
# Auto-reload edited .py files (so `git pull` changes to src/ take effect
# without restarting the kernel).
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.expanduser('~/mental-health/src'))
import numpy as np
from extract_eeg_embeddings import load_raw, preprocess, make_windows, extract_labram_embeddings

SCRATCH = os.environ.get('SCRATCH') or f"/scratch/{os.environ['USER']}"
SUBJECT = f'{SCRATCH}/mh/data/mipdb/A00051955/'
print('subject path:', SUBJECT)
print('exists:', os.path.isdir(SUBJECT))

In [ ]:
# Load -> LaBraM-spec preprocessing -> 15s windows
raw = load_raw(SUBJECT)
raw = preprocess(raw)
windows = make_windows(raw, window_sec=15.0)
print('windows shape (n_windows, n_channels, n_samples):', windows.shape)

In [ ]:
# Frozen pretrained LaBraM -> one 200-dim embedding per 15s window
emb = extract_labram_embeddings(windows, raw.info['chs'])
print('embeddings shape:', emb.shape)
emb[:2]

In [ ]:
# Save the embeddings to $SCRATCH (same output the command-line run produces)
out_path = f'{SCRATCH}/mh/data/processed/A00051955_labram.npz'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
np.savez_compressed(out_path, embeddings=emb)
print('saved', emb.shape, '->', out_path)

In [ ]:
# Sanity check: are the 24 windows actually DISTINGUISHABLE, or degenerate?
# Volt-scale (buggy) input tends to collapse every window to ~the same vector.
print(f'value range : [{emb.min():.3f}, {emb.max():.3f}]  mean={emb.mean():.3f}  std={emb.std():.3f}')

# How many of the 200 dims actually vary across windows (not dead/constant)?
dim_std = emb.std(axis=0)
alive = (dim_std > 1e-4).sum()
print(f'active dims  : {alive}/200 vary across windows')

# Pairwise cosine similarity between windows. ~1.000 everywhere = degenerate.
norm = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8)
cos = norm @ norm.T
off = cos[~np.eye(len(cos), dtype=bool)]
print(f'window similarity (off-diagonal cosine): mean={off.mean():.3f}  min={off.min():.3f}')
print('  -> mean well below 1.0 = windows are distinct (good).'
      ' All ~1.000 = degenerate (bad).')

In [ ]:
# Visualise the embeddings (windows x 200-dim LaBraM features)
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.imshow(emb, aspect='auto', cmap='viridis')
plt.xlabel('LaBraM embedding dimension'); plt.ylabel('15s window')
plt.title('A00051955 - frozen LaBraM EEG embeddings'); plt.colorbar()
plt.tight_layout(); plt.show()